# cAPTure: graph-training runner binding

This notebook binds the one-seed graph-training runner to the approved immutable selection materialization and complete-fold graph collection. It freezes the twelve model/fold jobs, exact architecture, optimization, weighting, early-stopping, final-refit, persistence, and pilot-decision contracts.

It does not instantiate a model, run inference, perform optimization, read an outer-validation score, or access Test1, Test2, `train_pub_exf`, or `train_user_prop`. A separate manual authorization, bound to the generated report and job-plan hashes, is required before any backward pass.

## 1. Mount Drive and prepare the environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_SELECTION_PARENT = Path("/content/capture_graph_selection_inputs")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_training_binding import (
    run_capture_graph_training_binding,
    save_graph_training_authorization,
    stage_capture_selection_materialization,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Training-runner binding environment is ready.")

## 2. Bind the approved immutable inputs

In [ ]:
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
TRAINING_PREFLIGHT_RUN_ID = (
    "20260929T131750_353987Z_graph_training_preflight_v2"
)
SELECTION_MATERIALIZATION_RUN_ID = (
    "20260929T134737_484622Z_graph_selection_materialization"
)
SELECTION_MATERIALIZATION_MANIFEST_SHA256 = (
    "846d21671d6c657903427fefe665e8d1919c74859703a068a4940d02d8c09633"
)

GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
TRAINING_PREFLIGHT_DIR = (
    DRIVE_ROOT / "graph_training_preflight_v2_runs" / TRAINING_PREFLIGHT_RUN_ID
)
SELECTION_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_selection_materialization_runs"
    / SELECTION_MATERIALIZATION_RUN_ID
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
PREFLIGHT_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_preflight_v2.yaml"
)
SELECTION_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_selection_materialization_v1.yaml"
)
TRAINING_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_v1.yaml"
)

for path in (
    GRAPH_MATERIALIZATION_DIR, TRAINING_PREFLIGHT_DIR,
    SELECTION_MATERIALIZATION_DIR, GRAPH_INPUT_CONTRACT_PATH,
    PREFLIGHT_CONFIG_PATH, SELECTION_CONFIG_PATH, TRAINING_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")

print("Approved selection materialization:", SELECTION_MATERIALIZATION_DIR)
print("Frozen training contract:", TRAINING_CONFIG_PATH)

## 3. Stage and verify both graph collections

The complete-fold collection supplies final-refit and outer-validation tensors. The selection collection supplies only the fold-training scenarios transformed by the inner-training-only preprocessors. Both collections are copied to local Colab storage before repeated access.

In [ ]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
selection_root, selection_manifest = stage_capture_selection_materialization(
    source_root=SELECTION_MATERIALIZATION_DIR,
    local_parent=LOCAL_SELECTION_PARENT,
    collection=collection,
    config_path=SELECTION_CONFIG_PATH,
    expected_run_id=SELECTION_MATERIALIZATION_RUN_ID,
    expected_manifest_sha256=SELECTION_MATERIALIZATION_MANIFEST_SHA256,
)
print("Verified complete-fold graphs:", collection.root)
print("Verified selection graphs:", selection_root)

## 4. Create or reuse the immutable runner binding

This step constructs metadata and verifies data ranges and weights. It does not instantiate or train a model.

In [ ]:
BINDING_RUN_ID = None  # Paste an existing completed ID to reuse it.
if BINDING_RUN_ID is None:
    BINDING_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_training_binding"
    )
BINDING_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_training_binding_runs" / BINDING_RUN_ID
)
print("Training-runner binding ID:", BINDING_RUN_ID)
print("Training-runner binding output:", BINDING_OUTPUT_DIR)

binding_report = run_capture_graph_training_binding(
    collection=collection,
    selection_root=selection_root,
    preflight_dir=TRAINING_PREFLIGHT_DIR,
    training_config_path=TRAINING_CONFIG_PATH,
    preflight_config_path=PREFLIGHT_CONFIG_PATH,
    selection_config_path=SELECTION_CONFIG_PATH,
    output_dir=BINDING_OUTPUT_DIR,
)
print(json.dumps(binding_report, indent=2))

## 5. Review the exact jobs, slices, and weights

In [ ]:
job_plan = json.loads(
    (BINDING_OUTPUT_DIR / "training_job_plan.json").read_text(encoding="utf-8")
)
display(pd.DataFrame(job_plan["jobs"]))

slice_rows = []
weight_rows = []
for fold, fold_plan in job_plan["fold_plans"].items():
    for scenario, sequence in fold_plan["selection_sequences"].items():
        slice_rows.append({"fold": fold, "scenario": scenario, **sequence})
        weight_rows.append({
            "fold": fold,
            "phase": "selection",
            "scenario": scenario,
            **fold_plan["selection_scenario_class_weights"][scenario],
        })
        weight_rows.append({
            "fold": fold,
            "phase": "complete_refit",
            "scenario": scenario,
            **fold_plan["complete_refit_scenario_class_weights"][scenario],
        })
display(pd.DataFrame(slice_rows))
display(pd.DataFrame(weight_rows))
display(pd.DataFrame.from_dict(job_plan["model_specifications"], orient="index"))
print("Binding status:", binding_report["status"])
print("Jobs frozen:", binding_report["jobs"])
print("Training performed:", binding_report["training_performed"])
print("Held-out scenarios accessed:", binding_report["held_out_scenarios_accessed"])

## 6. Optional manual authorization

Review the report, the twelve jobs, graph ranges, separate selection/refit weights, and frozen model parameters first. Leave `AUTHORIZE_MODEL_TRAINING` as `None` until the binding is accepted. This decision authorizes only development-fold graph optimization; it never authorizes Test1 or Test2.

In [ ]:
AUTHORIZE_MODEL_TRAINING = None
TRAINING_AUTHORIZATION_NOTE = (
    "Approved for the one-seed development graph pilot. The runner is bound "
    "to the approved selection materialization and complete-fold graph "
    "collection. The twelve model/fold jobs, inner-validation slices, "
    "scenario/class weights, architecture, optimizer, temporal-memory policy, "
    "early-stopping rule, complete-data refit, and OOF output contract are "
    "frozen. This authorization does not permit held-out-scenario access."
)

if AUTHORIZE_MODEL_TRAINING is None:
    print("No training authorization was written. Review the binding first.")
else:
    authorization = save_graph_training_authorization(
        BINDING_OUTPUT_DIR,
        approved=bool(AUTHORIZE_MODEL_TRAINING),
        review_note=TRAINING_AUTHORIZATION_NOTE,
    )
    print("Saved training authorization:", json.dumps(authorization, indent=2))